# Pipeline Demo — Phase 5

Demonstrates the new `portfolio.pipeline` module that replaces the
~80 lines of boilerplate duplicated across cost_analysis, validation,
risk_analysis, and alpha_iteration notebooks.

**Pipeline stages:**
1. `compute_daily_returns()` — OHLCV → daily return series
2. `compute_next_day_returns()` — daily returns → tradable next-day returns
3. `build_factor_pipeline()` — OHLCV → preprocessed composite factor
4. `build_sizing_methods()` — composite → 4 sizing method weights
5. `resample_weights()` — reduce rebalancing frequency
6. `compute_portfolio_return()` — weights × returns → portfolio returns
7. `run_alpha_pipeline()` — all-in-one wrapper

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px

from data.universe import get_universe
from data.loader.data_loader import stock_load_process
from constants import DATE_COL, TICKER_COL, VALUE_COL, WEIGHT_COL, TRADING_DAYS_PER_YEAR

from portfolio.pipeline import (
    compute_daily_returns,
    compute_next_day_returns,
    compute_portfolio_return,
    resample_weights,
    build_factor_pipeline,
    build_sizing_methods,
    run_alpha_pipeline,
)
from portfolio.alpha_config import AlphaConfig, FactorConfig
from factors.factors import get_factor_fn,list_factors
from risk.position_sizing import size_equal_weight, size_signal_weighted

available_factors = list_factors()
print(f"Available factors: {available_factors}")

Loading alpha101 module...
About to register factors...
Available factors: ['alpha001', 'alpha002', 'alpha003']


# Select Alpha Factor

In [3]:
import inspect

factor_name = available_factors[1]
# factor_name = available_factors[1]
print(f"Using factor: {factor_name}")

# list factors and their parameters
factor_cls = get_factor_fn(factor_name)
print(f"\nFactor parameters for {factor_name}:")
for name, p in inspect.signature(factor_cls).parameters.items():
    # if name in ("ohlcv", "kwargs"):
    if name in ("kwargs"):
        continue
    default = f" = {p.default}" if p.default is not inspect.Parameter.empty else ""
    print(f"  {name}{default}")

Using factor: alpha002

Factor parameters for alpha002:
  ohlcv


## 1. Load Data (same date range as diagnostics notebook)

In [4]:
UNIVERSE = get_universe("US_LARGE_CAP_50")
START_DATE = "2023-01-01"
END_DATE = "2026-02-28"

ohlcv = (
    stock_load_process(tickers=UNIVERSE, start_date=START_DATE, end_date=END_DATE)
    .filter(pl.col("volume") != 0)
    .collect()
)
print(f"OHLCV: {ohlcv.shape[0]:,} rows, {ohlcv.select('ticker').n_unique()} tickers")

# Precompute shared data
daily_returns = compute_daily_returns(ohlcv)
next_day_returns = compute_next_day_returns(daily_returns)
alpha_factor = get_factor_fn(factor_name)(
    ohlcv,
    # boll_length=50, # test
    factor_config=FactorConfig()
)

print(f"Daily returns: {daily_returns.shape[0]:,} rows")
print(f"Next-day returns: {next_day_returns.shape[0]:,} rows")
print(f"alpha factor: {alpha_factor.shape[0]:,} rows")

Loading from cache: /mnt/blackdisk/quant_data/polygon_data/processed/us_stocks_sip/day_aggs_v1/cache_c5e11349e9e04e8bdef2634ae48ac375.parquet
Cache loaded: 40,301 rows, 2.58 MB
OHLCV: 40,093 rows, 52 tickers
Daily returns: 40,041 rows
Next-day returns: 39,989 rows
alpha factor: 39,729 rows


## 2. Ablation Study — Sizing × Rebalancing

Test all 4 combinations to isolate what flips the Sharpe from +0.814 to -0.393.

In [5]:
n_long = 10
n_short = 10
# Helper: compute Sharpe from weights + returns
def sharpe_from_weights(weights, ndr, rebal_n=1):
    if rebal_n > 1:
        weights = resample_weights(weights, rebal_every_n=rebal_n)
    port_ret = compute_portfolio_return(weights, ndr)
    arr = port_ret["port_return"].to_numpy()
    mu = float(np.mean(arr))
    sigma = float(np.std(arr, ddof=1))
    sharpe = mu / sigma * np.sqrt(TRADING_DAYS_PER_YEAR) if sigma > 1e-10 else 0.0
    return sharpe, mu * TRADING_DAYS_PER_YEAR, sigma * np.sqrt(TRADING_DAYS_PER_YEAR), arr, port_ret

# --- Equal-Weight weights ---
ew_weights = size_equal_weight(alpha_factor, n_long=n_long, n_short=n_short)

# --- Signal-weighted weights ---
returns_for_sw = daily_returns.rename({"daily_return": "return"})
signal_weights = size_signal_weighted(
    alpha_factor, returns_for_sw,
    n_long=n_long, n_short=n_short,
    lookback=60, max_position=0.10, max_leverage=1.0,
)

print(f"EW weights: {ew_weights.shape[0]:,} rows")
print(f"Signal weights: {signal_weights.shape[0]:,} rows")

# --- 2×2 Ablation ---
configs = {
    "A: EW + Daily":   (ew_weights, 1),
    "B: EW + Weekly":  (ew_weights, 5),
    "C: Signal + Daily": (signal_weights, 1),
    "D: Signal + Weekly": (signal_weights, 5),
}

print(f"\n{'Config':<22} {'Sharpe':>8} {'AnnRet':>8} {'AnnVol':>8}")
print("-" * 50)
ablation_results = {}
for label, (w, rebal) in configs.items():
    s, ar, av, arr, pr = sharpe_from_weights(w, next_day_returns, rebal)
    ablation_results[label] = {"sharpe": s, "arr": arr, "port_ret": pr}
    print(f"{label:<22} {s:>+8.3f} {ar:>+7.2%} {av:>7.2%}")

EW weights: 15,580 rows
Signal weights: 14,469 rows

Config                   Sharpe   AnnRet   AnnVol
--------------------------------------------------
A: EW + Daily            -0.212  -1.42%   6.69%
B: EW + Weekly           -0.706  -4.75%   6.73%
C: Signal + Daily        -0.392  -2.29%   5.83%
D: Signal + Weekly       +0.360  +1.93%   5.36%


## 3. Equity Curves — Visual Comparison

In [7]:
colors = px.colors.qualitative.Set2
fig = go.Figure()
for i, (label, r) in enumerate(ablation_results.items()):
    cum = np.cumprod(1 + r["arr"])
    dates = r["port_ret"][DATE_COL].to_list()
    fig.add_trace(go.Scatter(
        x=dates, y=cum,
        name=f"{label} (S={r['sharpe']:+.2f})",
        line=dict(color=colors[i % len(colors)])
    ))
fig.add_hline(y=1.0, line_dash="dash", line_color="gray")
fig.update_layout(
    title=f"{factor_name} Ablation: Sizing × Rebalancing",
    yaxis_title="Cumulative Return",
    height=500,
)
fig.show()

## 4. Weight Diagnostics — Kelly vs Equal-Weight

Examine the weight distribution, net exposure, and concentration.

In [8]:
# --- Net exposure per date ---
def weight_stats(weights_df, label):
    stats = (
        weights_df.group_by(DATE_COL)
        .agg([
            pl.col(WEIGHT_COL).sum().alias("net_exposure"),
            pl.col(WEIGHT_COL).abs().sum().alias("gross_leverage"),
            pl.col(WEIGHT_COL).filter(pl.col(WEIGHT_COL) > 0).sum().alias("long_wt"),
            pl.col(WEIGHT_COL).filter(pl.col(WEIGHT_COL) < 0).sum().alias("short_wt"),
            pl.col(WEIGHT_COL).abs().max().alias("max_position"),
            pl.col(WEIGHT_COL).count().alias("n_positions"),
        ])
        .sort(DATE_COL)
    )
    print(f"\n--- {label} ---")
    print(f"  Dates: {stats.shape[0]}")
    print(f"  Mean net exposure:   {stats['net_exposure'].mean():+.4f}")
    print(f"  Mean gross leverage: {stats['gross_leverage'].mean():.4f}")
    print(f"  Mean long weight:    {stats['long_wt'].mean():+.4f}")
    print(f"  Mean short weight:   {stats['short_wt'].mean():+.4f}")
    print(f"  Mean max position:   {stats['max_position'].mean():.4f}")
    print(f"  Mean # positions:    {stats['n_positions'].mean():.1f}")
    return stats

ew_stats = weight_stats(ew_weights, "Equal-Weight")
signal_stats = weight_stats(signal_weights, "Signal-Weighted")


--- Equal-Weight ---
  Dates: 779
  Mean net exposure:   -0.0000
  Mean gross leverage: 1.0000
  Mean long weight:    +0.5000
  Mean short weight:   -0.5000
  Mean max position:   0.0500
  Mean # positions:    20.0

--- Signal-Weighted ---
  Dates: 726
  Mean net exposure:   +0.0231
  Mean gross leverage: 0.9163
  Mean long weight:    +0.4697
  Mean short weight:   -0.4466
  Mean max position:   0.0998
  Mean # positions:    19.9


## 5. Net Exposure Over Time — Is Kelly Creating Directional Bias?

In [9]:
# Plot net exposure over time
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ew_stats[DATE_COL].to_list(),
    y=ew_stats["net_exposure"].to_numpy(),
    name="EW net exposure",
    line=dict(color=colors[0]),
))
fig.add_trace(go.Scatter(
    x=signal_stats[DATE_COL].to_list(),
    y=signal_stats["net_exposure"].to_numpy(),
    name="Signal net exposure",
    line=dict(color=colors[1]),
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Net Exposure: EW vs Signal-Weighted (0 = market-neutral)",
    yaxis_title="Net Exposure (long + short weights)",
    height=400,
)
fig.show()

## 6. Spot-Check: Who's in Each Portfolio on a Specific Date?

Pick one date and compare which stocks are long/short and their weights.

In [10]:
# Pick a date that exists in both weight sets
common_dates = set(ew_weights[DATE_COL].to_list()) & set(signal_weights[DATE_COL].to_list())
sample_date = sorted(common_dates)[len(common_dates) // 2]  # middle of period
print(f"Sample date: {sample_date}\n")

# EW positions
ew_day = ew_weights.filter(pl.col(DATE_COL) == sample_date).sort(WEIGHT_COL, descending=True)
print("--- Equal-Weight Positions ---")
for row in ew_day.iter_rows(named=True):
    side = "LONG " if row[WEIGHT_COL] > 0 else "SHORT"
    print(f"  {side} {row[TICKER_COL]:>5}  w={row[WEIGHT_COL]:+.4f}")

# Kelly positions
kelly_day = signal_weights.filter(pl.col(DATE_COL) == sample_date).sort(WEIGHT_COL, descending=True)
print("\n--- Half-Kelly Positions ---")
for row in kelly_day.iter_rows(named=True):
    side = "LONG " if row[WEIGHT_COL] > 0 else "SHORT"
    print(f"  {side} {row[TICKER_COL]:>5}  w={row[WEIGHT_COL]:+.4f}")

# Check: are the same stocks in both?
ew_tickers = set(ew_day[TICKER_COL].to_list())
kelly_tickers = set(kelly_day[TICKER_COL].to_list())
print(f"\nSame stock selection? {ew_tickers == kelly_tickers}")
print(f"  EW-only: {ew_tickers - kelly_tickers}")
print(f"  Kelly-only: {kelly_tickers - ew_tickers}")

Sample date: 2024-09-10 00:00:00-04:00

--- Equal-Weight Positions ---
  LONG    JPM  w=+0.0500
  LONG    XOM  w=+0.0500
  LONG   NFLX  w=+0.0500
  LONG    RTX  w=+0.0500
  LONG     KO  w=+0.0500
  LONG   AVGO  w=+0.0500
  LONG     GS  w=+0.0500
  LONG  GOOGL  w=+0.0500
  LONG      C  w=+0.0500
  LONG    COP  w=+0.0500
  SHORT   PEP  w=-0.0500
  SHORT   MCD  w=-0.0500
  SHORT   MRK  w=-0.0500
  SHORT    DE  w=-0.0500
  SHORT  TSLA  w=-0.0500
  SHORT    PG  w=-0.0500
  SHORT    BA  w=-0.0500
  SHORT  AMZN  w=-0.0500
  SHORT   WMT  w=-0.0500
  SHORT   PFE  w=-0.0500

--- Half-Kelly Positions ---
  LONG     KO  w=+0.1000
  LONG    COP  w=+0.0692
  LONG    RTX  w=+0.0429
  LONG  GOOGL  w=+0.0397
  LONG    XOM  w=+0.0388
  LONG   NFLX  w=+0.0383
  LONG     GS  w=+0.0312
  LONG      C  w=+0.0308
  LONG    JPM  w=+0.0284
  LONG   AVGO  w=+0.0061
  SHORT  TSLA  w=-0.0063
  SHORT    BA  w=-0.0186
  SHORT  AMZN  w=-0.0206
  SHORT   MRK  w=-0.0393
  SHORT    DE  w=-0.0398
  SHORT   PFE  w=-0.0431

## 7. Weight-Signal Alignment Check

For Kelly to work correctly, the weights should be aligned with the factor:
- Stocks with HIGH factor → positive weight (long)
- Stocks with LOW factor → negative weight (short)
- Weight MAGNITUDE should track |z-score| / σ²

If Kelly is fighting the factor, we'll see it here.

In [11]:
# Join factor values with Kelly weights on the sample date
factor_day = alpha_factor.filter(pl.col(DATE_COL) == sample_date)
kelly_merged = (
    kelly_day
    .join(factor_day, on=[DATE_COL, TICKER_COL], how="left")
    .sort(WEIGHT_COL, descending=True)
)

print(f"{'Ticker':>6}  {'Weight':>8}  {'Factor':>8}  {'Side':>5}")
print("-" * 35)
for row in kelly_merged.iter_rows(named=True):
    side = "LONG" if row[WEIGHT_COL] > 0 else "SHORT"
    fv = row[VALUE_COL] if row[VALUE_COL] is not None else float("nan")
    print(f"{row[TICKER_COL]:>6}  {row[WEIGHT_COL]:+8.4f}  {fv:+8.3f}  {side:>5}")

# Check direction alignment
aligned = kelly_merged.with_columns(
    (pl.col(WEIGHT_COL) * pl.col(VALUE_COL)).alias("wf_product")
)
n_aligned = aligned.filter(pl.col("wf_product") > 0).shape[0]
n_total = aligned.shape[0]
print(f"\nDirection alignment: {n_aligned}/{n_total} positions have weight×factor > 0")

Ticker    Weight    Factor   Side
-----------------------------------
    KO   +0.1000    +1.227   LONG
   COP   +0.0692    +1.641   LONG
   RTX   +0.0429    +1.210   LONG
 GOOGL   +0.0397    +1.578   LONG
   XOM   +0.0388    +0.999   LONG
  NFLX   +0.0383    +1.166   LONG
    GS   +0.0312    +1.391   LONG
     C   +0.0308    +1.584   LONG
   JPM   +0.0284    +0.935   LONG
  AVGO   +0.0061    +1.272   LONG
  TSLA   -0.0063    -1.554  SHORT
    BA   -0.0186    -1.418  SHORT
  AMZN   -0.0206    -1.326  SHORT
   MRK   -0.0393    -1.786  SHORT
    DE   -0.0398    -1.562  SHORT
   PFE   -0.0431    -0.998  SHORT
   WMT   -0.0648    -1.314  SHORT
    PG   -0.0843    -1.509  SHORT
   MCD   -0.0868    -1.987  SHORT
   PEP   -0.1000    -1.987  SHORT

Direction alignment: 20/20 positions have weight×factor > 0


## 8. Date Coverage Diagnostic

Check if Kelly produces weights for all dates, or if many dates are missing (which would corrupt the backtest).

In [10]:
ew_dates = ew_weights.select(DATE_COL).unique().shape[0]
signal_dates = signal_weights.select(DATE_COL).unique().shape[0]
factor_dates = alpha_factor.select(DATE_COL).unique().shape[0]
ndr_dates = next_day_returns.select(DATE_COL).unique().shape[0]

print(f"Factor dates:        {factor_dates}")
print(f"Next-day ret dates:  {ndr_dates}")
print(f"EW weight dates:     {ew_dates}")
print(f"Signal weight dates: {signal_dates}")
print(f"Missing Signal dates: {factor_dates - signal_dates}")

# Check date range
ew_min = ew_weights[DATE_COL].min()
ew_max = ew_weights[DATE_COL].max()
signal_min = signal_weights[DATE_COL].min()
signal_max = signal_weights[DATE_COL].max()
print(f"\nEW range:    {ew_min} to {ew_max}")
print(f"Signal range: {signal_min} to {signal_max}")

Factor dates:        781
Next-day ret dates:  784
EW weight dates:     781
Signal weight dates: 726
Missing Signal dates: 55

EW range:    2023-01-10 00:00:00-05:00 to 2026-02-20 00:00:00-05:00
Signal range: 2023-03-30 00:00:00-04:00 to 2026-02-20 00:00:00-05:00


## 9. Full Pipeline Comparison (AlphaConfig API)

Run `run_alpha_pipeline()` with both sizing methods side-by-side.

In [11]:
configs = {
    f"{factor_name} EW daily": AlphaConfig(
        factor_names=[factor_name],
        sizing_method="Equal-Weight",
        rebal_every_n=1,
        name="alpha-EW-daily",
    ),
    f"{factor_name} EW weekly": AlphaConfig(
        factor_names=[factor_name],
        sizing_method="Equal-Weight",
        rebal_every_n=5,
        name="alpha-EW-weekly",
    ),
    f"{factor_name} Signal daily": AlphaConfig(
        factor_names=[factor_name],
        sizing_method="Signal-Weighted",
        rebal_every_n=1,
        name="alpha-Signal-daily",
    ),
    f"{factor_name} Signal weekly": AlphaConfig(
        factor_names=[factor_name],
        sizing_method="Signal-Weighted",
        rebal_every_n=5,
        name="alpha-Signal-weekly",
    ),
}

print(f"{'Config':<25} {'Sharpe':>8} {'AnnRet':>8} {'AnnVol':>8} {'Days':>6}")
print("-" * 60)
pipeline_results = {}
for label, cfg in configs.items():
    r = run_alpha_pipeline(ohlcv, config=cfg)
    pipeline_results[label] = r
    print(f"{label:<25} {r['sharpe']:>+8.3f} {r['annual_return']:>+7.2%} {r['annual_vol']:>7.2%} {r['n_days']:>6}")

Config                      Sharpe   AnnRet   AnnVol   Days
------------------------------------------------------------
alpha001 EW daily           +1.448 +26.01%  17.96%    780
alpha001 EW weekly          +1.471 +23.96%  16.28%    780
alpha001 Signal daily       +0.000   +nan%    nan%    725
alpha001 Signal weekly      +1.471 +15.64%  10.63%    725


## 10. Findings

| Config | Sharpe | Δ from A | Interpretation |
|--------|--------|----------|---------------|
| A: EW + Daily | **+0.814** | — | Baseline, matches diagnostics |
| B: EW + Weekly | +0.417 | -0.40 | **Rebalancing halves Sharpe** |
| C: Kelly + Daily | +0.137 | -0.68 | **Kelly destroys most alpha** |
| D: Kelly + Weekly | -0.393 | -1.21 | Both effects stack → negative |

### Two separate problems:

**Problem 1: Weekly rebalancing loses ~0.4 Sharpe** (A→B).
alpha is a high-frequency mean-reversion signal. Holding positions for 5 days
lets the mean-reversion effect decay. The signal needs daily rebalancing.

**Problem 2: Kelly sizing loses ~0.68 Sharpe** (A→C).
Kelly uses `|z|/σ²` which *concentrates* into low-vol stocks. alpha works because 
it identifies overbought/oversold across the *whole* cross-section equally — 
concentrating into low-vol names discards the high-vol names where alpha has the
strongest signal.  Also Kelly's `-0.0073 net exposure` introduces small directional bias.

### What's NOT wrong:
- **Stock selection is identical** (same 20 stocks in EW and Kelly)
- **Direction alignment is 100%** (factor → weight direction correct)
- **Date coverage is close** (726 vs 753 dates, only 27 missing)

### Implication:
The factor diagnostics notebook's L/S analysis (Section 4) uses Equal-Weight + Daily 
rebalancing — which IS the correct way to evaluate factor alpha. The pipeline's
default Half-Kelly + Weekly was destroying alpha, not adding to it.

For alpha specifically, **Equal-Weight sizing may be optimal** since the signal
strength is roughly uniform across the cross-section.